# Corrosion data cleaning

This notebook loads `data/corrosion_data.csv`, applies only the documented normalization and quality filters, produces `clean_data`, and exports the approved CSV and Parquet artifacts.

In [1]:
from pathlib import Path
import re

import pandas as pd
from pandas.testing import assert_frame_equal

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "pyproject.toml").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find project root containing pyproject.toml")

RAW_PATH = PROJECT_ROOT / "data" / "corrosion_data.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
CSV_PATH = OUTPUT_DIR / "corrosion_data_clean.csv"
PARQUET_PATH = OUTPUT_DIR / "corrosion_data_clean.parquet"


In [2]:
REQUIRED_SOURCE_COLUMNS = [
    "Paper ID", "Salt", "Temperature (celcius)", "Exposure Time (hours)",
    "Headgas", "Flow (m/s)", "Corrosion Rate (μm/yr)", "Alloy",
    "Ni (wt.%)", "Cr (wt.%)", "Fe (wt.%)", "C (wt.%)", "Mn (wt.%)",
    "Si (wt.%)", "N (wt.%)", "Nb (wt.%)", "Co (wt.%)", "Mo (wt.%)",
    "Al (wt.%)", "Ti (wt.%)", "Cu (wt.%)", "W (wt.%)", "P (wt.%)",
    "S (wt.%)", "Composition Check",
]

raw_data = pd.read_csv(RAW_PATH)
missing_columns = [column for column in REQUIRED_SOURCE_COLUMNS if column not in raw_data.columns]
if missing_columns:
    raise ValueError(f"Missing required source columns: {missing_columns}")

print("Raw shape:", raw_data.shape)
print("Raw data types:\n", raw_data.dtypes)
print("Missing counts greater than zero:\n", raw_data.isna().sum().loc[lambda counts: counts.gt(0)])
print("Exact duplicate count:", raw_data.duplicated().sum())
for column in ["Salt", "Headgas", "Alloy"]:
    print(f"Unique {column} values:", raw_data[column].unique())


Raw shape: (116, 25)
Raw data types:
 Paper ID                      str
Salt                          str
Temperature (celcius)       int64
Exposure Time (hours)     float64
Headgas                       str
Flow (m/s)                float64
Corrosion Rate (μm/yr)        str
Alloy                         str
Ni (wt.%)                 float64
Cr (wt.%)                 float64
Fe (wt.%)                 float64
C (wt.%)                      str
Mn (wt.%)                     str
Si (wt.%)                 float64
N (wt.%)                  float64
Nb (wt.%)                 float64
Co (wt.%)                 float64
Mo (wt.%)                 float64
Al (wt.%)                 float64
Ti (wt.%)                 float64
Cu (wt.%)                 float64
W (wt.%)                  float64
P (wt.%)                      str
S (wt.%)                      str
Composition Check         float64
dtype: object
Missing counts greater than zero:
 Exposure Time (hours)     2
Headgas                   2
Flow (m

In [3]:
data = raw_data.copy()
ELEMENT_COLUMNS = [
    "Ni (wt.%)", "Cr (wt.%)", "Fe (wt.%)", "C (wt.%)", "Mn (wt.%)",
    "Si (wt.%)", "N (wt.%)", "Nb (wt.%)", "Co (wt.%)", "Mo (wt.%)",
    "Al (wt.%)", "Ti (wt.%)", "Cu (wt.%)", "W (wt.%)", "P (wt.%)",
    "S (wt.%)",
]

def parse_composition_value(value):
    if pd.isna(value):
        return float("nan")

    text = str(value).strip()
    if text.startswith("≤"):
        text = text[1:].strip()
    elif text.startswith("<="):
        text = text[2:].strip()

    range_match = re.fullmatch(
        r"([0-9]+(?:\.[0-9]+)?)\s*-\s*([0-9]+(?:\.[0-9]+)?)",
        text,
    )
    if range_match:
        lower, upper = map(float, range_match.groups())
        return (lower + upper) / 2

    return float(text)

assert parse_composition_value("≤0.27") == 0.27
assert parse_composition_value("<=0.025") == 0.025
assert parse_composition_value("0.85-1.2") == 1.025

data[ELEMENT_COLUMNS] = data[ELEMENT_COLUMNS].map(parse_composition_value)
data["Corrosion Rate (μm/yr)"] = pd.to_numeric(
    data["Corrosion Rate (μm/yr)"].astype("string").str.strip().str.replace(r"^<", "", regex=True),
    errors="raise",
)
data["Alloy"] = data["Alloy"].astype("string").str.strip()


In [4]:
ESSENTIAL_COLUMNS = [
    "Temperature (celcius)", "Exposure Time (hours)", "Headgas", "Flow (m/s)",
    "Corrosion Rate (μm/yr)", "Alloy", *ELEMENT_COLUMNS,
]
missing_essential_mask = data[ESSENTIAL_COLUMNS].isna().any(axis=1)
print("Rows with missing essential values:", missing_essential_mask.sum())
data = data.loc[~missing_essential_mask].copy()

data["recomputed_composition_total"] = data[ELEMENT_COLUMNS].sum(axis=1)
composition_mask = data["recomputed_composition_total"].between(95, 105, inclusive="both")
print("Rows rejected by recomputed composition total:\n", data.loc[~composition_mask, [
    "Paper ID", "Alloy", "recomputed_composition_total"
]])
data = data.loc[composition_mask].copy()


Rows with missing essential values: 2
Rows rejected by recomputed composition total:
    Paper ID                   Alloy  recomputed_composition_total
15     P003                    316L                        36.170
23     P007  A516 Gr70 Carbon Steel                         1.345
24     P007  A516 Gr70 Carbon Steel                         1.345
25     P007  A516 Gr70 Carbon Steel                         1.345
26     P007  A516 Gr70 Carbon Steel                         1.345


In [5]:
salt_values = data["Salt"].dropna().unique()
if len(salt_values) != 1:
    raise ValueError(f"Expected exactly one non-null Salt value, found {salt_values.tolist()}")
print("Salt value:", salt_values[0])

expected_headgas = {"air", "N2", "O2 (sealed loop)"}
actual_headgas = set(data["Headgas"].dropna().unique())
if actual_headgas != expected_headgas:
    raise ValueError(f"Headgas categories differ: expected {expected_headgas}, actual {actual_headgas}")

headgas_columns = ["headgas_air", "headgas_n2", "headgas_o2_sealed_loop"]
headgas_dummies = pd.get_dummies(data["Headgas"], dtype="int8").rename(columns={
    "air": "headgas_air",
    "N2": "headgas_n2",
    "O2 (sealed loop)": "headgas_o2_sealed_loop",
})[headgas_columns]
data = pd.concat([data.drop(columns=["Salt", "Headgas", "Composition Check", "recomputed_composition_total"]), headgas_dummies], axis=1)

EXPECTED_COLUMNS = [
    "paper_id", "alloy", "temperature_c", "exposure_time_h", "flow_m_per_s",
    "corrosion_rate_um_per_year", "headgas_air", "headgas_n2", "headgas_o2_sealed_loop",
    "ni_wt_pct", "cr_wt_pct", "fe_wt_pct", "c_wt_pct", "mn_wt_pct", "si_wt_pct",
    "n_wt_pct", "nb_wt_pct", "co_wt_pct", "mo_wt_pct", "al_wt_pct", "ti_wt_pct",
    "cu_wt_pct", "w_wt_pct", "p_wt_pct", "s_wt_pct",
]
COLUMN_RENAMES = {
    "Paper ID": "paper_id", "Alloy": "alloy", "Temperature (celcius)": "temperature_c",
    "Exposure Time (hours)": "exposure_time_h", "Flow (m/s)": "flow_m_per_s",
    "Corrosion Rate (μm/yr)": "corrosion_rate_um_per_year",
    "Ni (wt.%)": "ni_wt_pct", "Cr (wt.%)": "cr_wt_pct", "Fe (wt.%)": "fe_wt_pct",
    "C (wt.%)": "c_wt_pct", "Mn (wt.%)": "mn_wt_pct", "Si (wt.%)": "si_wt_pct",
    "N (wt.%)": "n_wt_pct", "Nb (wt.%)": "nb_wt_pct", "Co (wt.%)": "co_wt_pct",
    "Mo (wt.%)": "mo_wt_pct", "Al (wt.%)": "al_wt_pct", "Ti (wt.%)": "ti_wt_pct",
    "Cu (wt.%)": "cu_wt_pct", "W (wt.%)": "w_wt_pct", "P (wt.%)": "p_wt_pct",
    "S (wt.%)": "s_wt_pct",
}
clean_data = data.rename(columns=COLUMN_RENAMES)[EXPECTED_COLUMNS].copy()
clean_data[["paper_id", "alloy"]] = clean_data[["paper_id", "alloy"]].astype("string")


Salt value: 0.40 KNO3 + 0.60 NaNO3


In [6]:
from pandas.api.types import is_numeric_dtype

assert clean_data.columns.tolist() == EXPECTED_COLUMNS
assert not clean_data.isna().any().any()
assert clean_data[[
    "headgas_air", "headgas_n2", "headgas_o2_sealed_loop"
]].sum(axis=1).eq(1).all()
for column in set(EXPECTED_COLUMNS) - {"paper_id", "alloy"}:
    assert is_numeric_dtype(clean_data[column]), f"Expected numeric dtype for {column}"

print("Exact duplicate count in clean data:", clean_data.duplicated().sum())
print("Rows with temperature above 1000 C:\n", clean_data.loc[clean_data["temperature_c"] > 1000])
print("Final shape:", clean_data.shape)
print("Final data types:\n", clean_data.dtypes)


Exact duplicate count in clean data: 0
Rows with temperature above 1000 C:
    paper_id                     alloy  temperature_c  exposure_time_h  \
49     P008  Type 316 Stainless Steel           4765           4500.0   

    flow_m_per_s  corrosion_rate_um_per_year  headgas_air  headgas_n2  \
49           0.0                         3.0            0           0   

    headgas_o2_sealed_loop  ni_wt_pct  ...  n_wt_pct  nb_wt_pct  co_wt_pct  \
49                       1       11.0  ...       0.0        0.0        0.0   

    mo_wt_pct  al_wt_pct  ti_wt_pct  cu_wt_pct  w_wt_pct  p_wt_pct  s_wt_pct  
49        2.0        0.0        0.0        0.0       0.0       0.0       0.0  

[1 rows x 25 columns]
Final shape: (109, 25)
Final data types:
 paper_id                       string
alloy                          string
temperature_c                   int64
exposure_time_h               float64
flow_m_per_s                  float64
corrosion_rate_um_per_year    Float64
headgas_air           

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean_data.to_csv(CSV_PATH, index=False)
clean_data.to_parquet(PARQUET_PATH, index=False)

csv_reloaded = pd.read_csv(CSV_PATH)
parquet_reloaded = pd.read_parquet(PARQUET_PATH)
assert_frame_equal(
    csv_reloaded,
    parquet_reloaded,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)
print(f"Exported CSV: {CSV_PATH}")
print(f"Exported Parquet: {PARQUET_PATH}")
print("Exported shape:", clean_data.shape)


Exported CSV: /home/mhwkm/projects/molten_salt/data/processed/corrosion_data_clean.csv
Exported Parquet: /home/mhwkm/projects/molten_salt/data/processed/corrosion_data_clean.parquet
Exported shape: (109, 25)
